# HartmannTwoWay3D — 3D two-way Hartmann flow

3D twin of `MagneticCoupled/HartmannTwoWay` (baseline Ha=5 case, z-extruded).


## Introduction

3D extrusion of the two-way Hartmann channel. The analytic profile is z-independent; the case runs in the mid-plane z=0.


In [ ]:
from trustutils import run
run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read TRUST probe (`.son`) file


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile
BUILD = run.BUILD_DIRECTORY
def read_son(case, name):
    cands = [f"{BUILD}/coarse/{case}_{name}.son", f"{BUILD}/{case}_{name}.son"]
    path = next((p for p in cands if os.path.exists(p)), cands[-1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)


## Run


In [ ]:
NB_PROCS = 2
run.reset()
NY0 = 65
meshes = {
    "coarse": {"nx": 5, "ny": NY0, "nz": 5},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate("hartmann_two_way_3d.data", lvl, dic=dic,
                                         nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()
run.printCases()
run.runCases()
case = cases["coarse"]


## Checks


In [ ]:
# applied transverse field invariant
t, by = read_son(case.dataFileName_, "SONDE_BY")
print(f"b_y last = {by[-1]:.6g}  (expected ~1)")
assert abs(by[-1]-1.0) < 0.1
# induced field finite
t, bx = read_son(case.dataFileName_, "SONDE_B0")
print(f"b_x last = {bx[-1]:.6g}")
assert np.all(np.isfinite(bx))
# velocity finite and suppressed
t, u = read_son(case.dataFileName_, "SONDE_U0")
print(f"u_x last = {u[-1]:.6g}")
assert np.all(np.isfinite(u))
print("OK: HartmannTwoWay3D run completed.")


## Computer Performance


In [ ]:
run.tablePerf()